# Transformers, Attention, and Pre-training

Speaker: Josh Nguyen

University of Pennsylvania

joshtn@engineering.upenn.edu

[Slides](https://docs.google.com/presentation/d/1kQIM5h_LA3NfwnA5HvyjyqLltR0y2UkW6fzy7juBwy8/edit?usp=sharing)

## 1. What is pre-training?

**Pre-training** refers to the process of training a language model for the task of **language modeling**.

Well, this isn't a useful definition, so we will use an example to illustrate it.

### 1.1. An example

Take a look at these sentences.

> In the beginning the universe was created.

> Happy families are all alike; every unhappy family is unhappy in its own way.

> Intelligent life on a planet comes of age when it first works out the reason for its own existence.

They sound quite different from these sentences.

> Colorless green ideas sleep furiously.

> Contextual nuances highlights the recursive tension inherent to the framework.

> True value lies in embracing the multifaceted tapestry of perspectives.

To **model** language is to have a way of telling how **likely** a sentence, like one of these, would or would not belong to a human language like English.

We already have some (informal) model of language in our heads:

- If a sentence is ungrammatical, it's probably not human language because we speak and write according to formal grammatical rules.
- If a sentence is semantically empty (it conveys nothing really), it's probably not human language because when we speak and write, we do so with semantic intent.

Cognitive scientists call these abilities **formal linguistic competence**: to produce and comprehend human language.

### 1.2. Language modeling

**Language modeling** is our attempt to teach AI to have formal linguistic competence. We want our AI models to:

- Learn and recognize what is human language and what is not.
- Quantify the uncertainty in their recognition.
- Generate plausible sentences of human language from what they have learned.

The way we do that is to use **probability**. In particular, how likely is it that the sentence

> The archive preserves the dignified distance between form and memory.

was written by a human? In other words, given a sentence $s$, we ask

$$
P(s) = ?
$$

Ideally, a human-generated sentence would have a higher $P(s)$ than one that's not.

This probability $P(s)$ represents our confidence in recognizing human language from the rest. The task of **language modeling** is to train a model to estimate this probability from data.

### 1.3. Sequence modeling

In language modeling, we represent a sentence (or a paragraph, a passage, etc.) as a **sequence of tokens**. The process of breaking a sentence down into tokens is called **tokenization**.

There are many ways to define what a token can be, but the easiest way is to think of them as *words*.

So, the sentence

> $s = $ "In the beginning the universe was created."

is transformed into

$$
s = \texttt{("In", "the", "beginning", "the", "universe", "was", "created", ".")},
$$

which is a sequence of 8 tokens.

Recall that we need to find the likelihood of this sentence. Is is now equal to

$$
P(s) = P(\texttt{"In", "the", "beginning", "the", "universe", "was", "created", "."}).
$$


### 1.4. Causal sequence modeling

OK, great. But how do we actually assign a number to

$$
P(s) = P(\texttt{"In", "the", "beginning", "the", "universe", "was", "created", "."}).
$$

Most modern LLMs treat this task as **causal sequence modeling**.

Essentially, they posit that in generating this sentence, a human does so **one token at a time, from left to right**.

- First, **generate** `In`. How likely is it that *any* sentence in English would start with the word `In`?
- Then, **given** `In`, **generate** `the`. How likely is it that *any* sentence in English which starts with `In` would be followed by the word `the`?
- Then, **given** `In the`, **generate** `beginning`. How likely is it that *any* sentence in English which starts with `In the` would be followed by the word `beginning`?
- And so on...until we finish the sentence.

Formally, given a sequence $s = (X_1, X_2, \ldots, X_n)$, where every $X_i$ is a token, causal language modeling breaks its probability down to a product of (conditional) probabilities:

$$
\begin{align*}
P(X_1, X_2, \ldots, X_n) = \quad & P(X_1) \\
& \times P(X_2 \mid X_1) \\
& \times P(X_3 \mid X_1, X_2) \\
& \ldots \\
& \times P(X_n \mid X_1, \ldots, X_{n-1}). \\
\end{align*}
$$

In short,

$$
P(X_1, X_2, \ldots, X_n) = \prod_{i=1}^{n} P(X_i \mid X_{1:i-1}).
$$

### 1.5. Back to pretraining

Given this formalism, we can ask ourselves this.

> OK, how do actually we estimate the probability $P(\texttt{"In"})$?

We can
- Go out there and find all sentences in English we can find.
- Count the number times a sentence actually starts with the word `In`.
- Normalize this count to give us an estimate of $P(\texttt{"In"})$.


Similarly,

> What is the probability of $P(\texttt{"the"} \mid \texttt{"In"})$?

The process is the same, but the normalization is a little different. We can:
- Go out there and find all sentences in English we can find **that start with `In`.**
- From these sentences, count the number of times that the second word is `the`.
- Normalize this count to give us an estimate of $P(\texttt{"the"} \mid \texttt{"In"})$.

Continue this until we are finished with the sentence.

**The core problem** here is that explicitly representing all of these conditional probabilities is **very expensive**, primarily because of the combinatoric blowup of the representation space.

**The solution:** We will teach an LLM to **learn** this probability in some statistical way, and hopefully it will give us a good approximation of this probability. **Pretraining** refers to the process of teaching this LLM to estimate this probability by exposing it to many, many, many example sentences.

## 2. Evaluating a language model

Regardless of its underlying architecture, how do we know **if our model is fitted well to our data**?

### 2.1. Likelihood

Recall that our model takes as input a sequence of tokens $s = (X_1, \ldots, X_n)$ and outputs a probability $P(s)$ that represents how likely this sequence is an example of human language.

Ideally, sentences genuinely belonging to human language would have high $P(s)$.

So, to determine the goodness of fit, we will give the model a real sentence $s$, and simply get its output probability $P(s)$ and see how high it is.

Often times we transform the likelihood $P(s)$ using the log function, and thus we have a measure called **log-likelihood**, $\log P(s)$.

Sometimes we may want to transform it further, into the **negative log-likelihood (NLL)**, $- \log P(s)$. Minimizing NLL is equivalent to maximizing the (log-)likelihood.

### 2.2. Perplexity

The input $s$ can be of **arbitrary length**. Most sentences are short, of about 15 to 20 words, but some can be very long. Nothing about the generative grammar of English prevents you from coming up with a sentence that's one order of magnitude longer.

**The problem here is that longer sentences have lower probabilities** simply because they are products of many more conditional probabilities.

**The solution**: We will normalize the log-likelihood so that it becomes a **per-token metric** and doesn't simply favor shorter sentences. This is what **perplexity** measures.

Suppose we have a sequence $s = (X_1, \ldots, X_n)$ of length $n$.

Recall that our causal language model $P$ gives us:

$$
\begin{align*}
P(s) &= \prod_{i=1}^{n} P(X_i \mid X_{1:i-1}) \\
&= P(X_1) \times P(X_2 \mid X_1) \times \ldots \times P(X_n \mid X_{1:n-1}).
\end{align*}
$$

To turn this into perplexity, first we turn it into **log-likelihood**

$$
\begin{align*}
\log P(s) &= \log \left[ P(X_1) \times P(X_2 \mid X_1) \times \ldots \times P(X_n \mid X_{1:n-1}) \right]. \\
&= \log P(X_1) + \log P(X_2 \mid X_1) + \ldots + \log P(X_n \mid X_{1:n-1}) \\
&= \sum_{i = 1}^{n} \log P(X_i \mid X_{1:i-1}).
\end{align*}
$$

So, essentially, we just log-transform each conditional probability, and then sum them together.

Now, we normalize the log-likelihood by the length of the sequence to get a **per-token** metric.

$$
\begin{align*}
\frac{1}{n} \log P(s) &= \frac{1}{n} \sum_{i = 1}^{n} \log P(X_i \mid X_{1:i-1}).
\end{align*}
$$

Then, applying the exponentiation to the negative of this result gives us the **perplexity** score of this sequence:

$$
\begin{align*}
\text{PPL}(s) = \exp\left( - \frac{1}{n} \log P(s) \right).
\end{align*}
$$

Perplexity is the measure of goodness of fit that we will use to evaluate our language models. Lower perplexity means higher per-token likelihood, and thus better fit to our data.

### 2.3. Example of perplexity

Here we will go through an exercise of calculating the perplexity of a sequence.

#### 2.3.1. Load a pre-trained model



In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Model name
# Source: https://huggingface.co/allenai/OLMo-1B-0724-hf
model_id = "allenai/OLMo-1B-0724-hf"

# Load model and its tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.float32,
    device_map="auto"
)
model.eval()

config.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.37k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.12M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/9.25k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/114 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

OlmoForCausalLM(
  (model): OlmoModel(
    (embed_tokens): Embedding(50304, 2048, padding_idx=1)
    (layers): ModuleList(
      (0-15): 16 x OlmoDecoderLayer(
        (self_attn): OlmoAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (v_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): OlmoMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): OlmoLayerNorm()
        (post_attention_layernorm): OlmoLayerNorm()
      )
    )
    (norm): OlmoLayerNorm()
    (rotary_emb): OlmoRotaryEmbedding()
  )
  (

#### 2.3.2. Pass an input sequence through the model

In [ ]:
sentence = "In the beginning the universe was created."

# Tokenize the sentence into tokens
inputs = tokenizer(sentence, return_tensors="pt").to(model.device)

In [ ]:
# Length of the sequence
token_ids = inputs["input_ids"][0].tolist()
n = len(token_ids)
print("Length of sequence:", n, "tokens", "\n")

# Print all tokens in this sentence out.
for idx, token_id in enumerate(token_ids):
    # The raw token
    raw_token = tokenizer.convert_ids_to_tokens(token_id)

    # The more easily readable token
    decoded = tokenizer.decode(token_id)

    print(f"{idx:<6} | {token_id:<10} | {raw_token:<16} | {repr(decoded)}")

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPTNeoXTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


Length of sequence: 8 tokens 

0      | 688        | In               | 'In'
1      | 253        | Ġthe             | ' the'
2      | 5068       | Ġbeginning       | ' beginning'
3      | 253        | Ġthe             | ' the'
4      | 10325      | Ġuniverse        | ' universe'
5      | 369        | Ġwas             | ' was'
6      | 3562       | Ġcreated         | ' created'
7      | 15         | .                | '.'


In [ ]:
# Get model outputs
with torch.no_grad():
    outputs = model(**inputs, labels=inputs["input_ids"])

# Logit scores are scores before softmax transformation
# [0] because our batch only has 1 example
logits = outputs.logits[0]

What does `logits` do?

- Basically, it is a matrix of 8 rows, representing 8 tokens in our input.
- Each row represents the logit scores for **all possible next tokens**. Since the vocabulary of OLMo has 50,304 tokens, **each row has 50,304 logit scores**.
- So, if we're on **row $i$**, the logit scores will give us the likelihood of **every possible token in the vocabulary** that can be at position **$i+1$**.
- Thus, we need to make sure that:
    - The logit scores are turned into log-likelihood scores.
    - We find the actual next token at position $i+1$ in our example sentence.

#### 2.3.3. Extract token likelihoods and perplexity from the model

**Important**: For a sequence $s = (X_1, \ldots, X_n)$ of length $n$, the actual implementation of a language model only calculates the logit scores for the **next token**.

- We actually do not have $P(X_1)$, which is the marginal probability.
- As a result, the calculation of perplexity is actually only done for $X_2$ to $X_n$. So the normalization by the sequence length is only done for $n-1$ instead of $n$ tokens.

There is a practical solution to this, which is to "pad" the sentence with a dummy token `<BOS>` (beginning of sentence), so that $X_1$ becomes the second token. However, we're skipping that implementation here.

In [ ]:
# Accumulate all token likelihoods in this variable
loglikelihood = 0

# For a sequence of n, we only have conditional probability for
# n - 1 "next" tokens
print(f"Number of tokens used for calculation:", len(token_ids) - 1)

print("\nActual likelihood of each token")

for i in range(len(token_ids) - 1):
    # Get the logit scores for all possible next tokens
    logits_all_next_tokens = logits[i]

    # Turn logit scores into log likelihood
    logp_all_next_tokens = torch.log_softmax(logits[i], dim=0)

    # Choose the loglikelihood of the actual next token (X_i)
    idx_next_token = token_ids[i + 1]
    next_token_text = tokenizer.decode(idx_next_token)
    log_p_next_token_given_context = logp_all_next_tokens[idx_next_token]

    # Get the context up until this next token (X_{1:i-1})
    context_tokens = token_ids[:i + 1]
    context_text = tokenizer.decode(context_tokens)

    print(f"    logP({next_token_text} | {context_text}) = {log_p_next_token_given_context:.4f}")
    loglikelihood += log_p_next_token_given_context

# Normalize negative log-likelihood and exponentiate to get perplexity
perplexity = torch.exp(-loglikelihood / (len(token_ids) - 1))

print(f"\nLog likelihood: {loglikelihood.item():.4f}")
print(f"Perplexity: {perplexity:.4f}")

Number of tokens used for calculation: 7

Actual likelihood of each token
    logP( the | In) = -1.8271
    logP( beginning | In the) = -4.8863
    logP( the | In the beginning) = -3.5942
    logP( universe | In the beginning the) = -6.0115
    logP( was | In the beginning the universe) = -0.7214
    logP( created | In the beginning the universe was) = -0.5190
    logP(. | In the beginning the universe was created) = -0.4145

Log likelihood: -17.9740
Perplexity: 13.0360


**So, the perplexity of this sentence is 13.0360**.

Now that we have this, we can actually calculate it faster by exponentiating the loss value of this output.

In [ ]:
# The loss is just the average negative log likehood
loss = outputs.loss.item()
print(f"Loss: {loss:.4f}")

# Exponentiating the loss gives us the perplexity
print(f"Perplexity: {torch.exp(outputs.loss).item():.4f}")

Loss: 2.5677
Perplexity: 13.0360


As we have stated, we want to minimize perplexity to get the best fit to our data. The model's loss function is just the logarithm of this perplexity, which is thus minimized as well.